# RAG v0 exploration

This notebook explores the Amazon Bedrock knowledge settings to use as a baseline for the production information RAG system.

## Summary

This notebook builds and measures a baseline Bedrock Knowledge Base over the product information in `data/knowledge_base/`:

1. **Setup:** checks AWS credentials and the local knowledge base folder.
2. **Part 1–3: Load the data.** Creates a private S3 bucket with a random suffix, syncs the Markdown files (product pages, FAQ, hair-type guide) into it, and checks that all 11 files arrived.
3. **Part 4: Retrieve and evaluate.** Defines an 18-question eval set (9 lexical and 9 paraphrased questions, each with its relevant document IDs). The `retrieve` function wraps the Bedrock `retrieve` API and returns hits in the shape `run()` expects. `run()` executes every question, and `metrics_at()` reports hit@1, hit@k, recall@k, MRR, and approximate context tokens.
4. **Cleanup:** the final cell deletes the bucket. It is commented out so it only runs on purpose.

Current result: on the k=10 run, the baseline scores 1.0 for hit@1, hit@k, recall@k, and MRR at k=5, with about 1,917 approximate context tokens.

In [14]:
import subprocess
import uuid
import pandas as pd
from pathlib import Path
import boto3

In [15]:
REGION = "us-east-1"
BUCKET_PREFIX = "product-information-kb-data"  # bucket names are global, so a suffix is added below
KB_DIR = Path("../data/knowledge_base").resolve()
KB_PREFIX = "knowledge_base/"             # key prefix inside the bucket

def run_aws(*args):
    """Run an aws CLI command and return its stdout. Raises if the command fails."""
    result = subprocess.run(["aws", *args], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"aws {' '.join(args)} failed:\n{result.stderr}")
    return result.stdout

print("Region:", REGION)
print("Knowledge base dir:", KB_DIR, KB_DIR.exists())
print(run_aws("sts", "get-caller-identity", "--output", "text"))

Region: us-east-1
Knowledge base dir: /Users/lilianabadillo/Projects/repo/customer-service-assistant/data/knowledge_base True
865780386937	arn:aws:iam::865780386937:user/admin-user	AIDA4TFEIJR4SOJNKACJT



## Part 1. Create the bucket

The suffix makes the name unique. Save the printed name and reuse it in later cells instead of generating a new one.

In [5]:
BUCKET = f"{BUCKET_PREFIX}-{uuid.uuid4().hex[:8]}"

create_args = ["s3api", "create-bucket", "--bucket", BUCKET, "--region", REGION]
if REGION != "us-east-1":
    # us-east-1 rejects a LocationConstraint; every other region requires it.
    create_args += ["--create-bucket-configuration", f"LocationConstraint={REGION}"]

print(run_aws(*create_args))
print("Created bucket:", BUCKET)

{
    "Location": "/product-information-kb-data-820eb591"
}

Created bucket: product-information-kb-data-820eb591


In [6]:
# Optional: block public access, which is the default for a private knowledge base.
#run_aws(
#    "s3api", "put-public-access-block",
#    "--bucket", BUCKET,
#    "--public-access-block-configuration",
#    "BlockPublicAcls=true,IgnorePublicAcls=true,BlockPublicPolicy=true,RestrictPublicBuckets=true",
#)
#print("Public access blocked for", BUCKET)

## Part 2. Load the files

`aws s3 sync` uploads only new or changed files, so it is safe to re-run after editing the knowledge base.
Only Markdown files are uploaded, which skips `.DS_Store` and other OS files.

In [7]:
s3_uri = f"s3://{BUCKET}/{KB_PREFIX}"

output = run_aws(
    "s3", "sync", str(KB_DIR), s3_uri,
    "--exclude", "*",
    "--include", "*.md",
    "--region", REGION,
)
print(output or "Nothing to upload (already in sync).")

Completed 1.8 KiB/18.4 KiB (11.5 KiB/s) with 11 file(s) remaining
upload: ../data/knowledge_base/faq.md to s3://product-information-kb-data-820eb591/knowledge_base/faq.md
Completed 1.8 KiB/18.4 KiB (11.5 KiB/s) with 10 file(s) remaining
Completed 3.1 KiB/18.4 KiB (10.2 KiB/s) with 10 file(s) remaining
upload: ../data/knowledge_base/products/weekly-repair-mask.md to s3://product-information-kb-data-820eb591/knowledge_base/products/weekly-repair-mask.md
Completed 3.1 KiB/18.4 KiB (10.2 KiB/s) with 9 file(s) remaining
Completed 5.0 KiB/18.4 KiB (15.1 KiB/s) with 9 file(s) remaining
upload: ../data/knowledge_base/hair-type-guide.md to s3://product-information-kb-data-820eb591/knowledge_base/hair-type-guide.md
Completed 5.0 KiB/18.4 KiB (15.1 KiB/s) with 8 file(s) remaining
Completed 6.3 KiB/18.4 KiB (18.8 KiB/s) with 8 file(s) remaining
upload: ../data/knowledge_base/products/heat-shield-mist.md to s3://product-information-kb-data-820eb591/knowledge_base/products/heat-shield-mist.md
Comple

In [8]:
# Starter helper for uploading one file at a time, for example after editing a single document.
#def upload_file(local_path: Path) -> str:
#    key = KB_PREFIX + local_path.relative_to(KB_DIR).as_posix()
#    run_aws("s3", "cp", str(local_path), f"s3://{BUCKET}/{key}", "--region", REGION)
#    return key

# Example: re-upload one document
# upload_file(KB_DIR / "faq.md")

## Part 3. Verify the upload

List the objects in the bucket and compare the count with the local Markdown files.

In [9]:
listing = run_aws("s3", "ls", f"s3://{BUCKET}/{KB_PREFIX}", "--recursive", "--human-readable", "--summarize")
print(listing)

local_count = len(list(KB_DIR.rglob("*.md")))
print("Local .md files:", local_count)

2026-11-19 11:52:48    4.0 KiB knowledge_base/aws-developers-discount.md
2026-11-19 11:52:48    1.8 KiB knowledge_base/faq.md
2026-11-19 11:52:48    1.9 KiB knowledge_base/hair-type-guide.md
2026-11-19 11:52:48    1.3 KiB knowledge_base/products/air-dry-styling-gel.md
2026-11-19 11:52:48    1.4 KiB knowledge_base/products/define-curl-cream.md
2026-11-19 11:52:48    1.3 KiB knowledge_base/products/heat-shield-mist.md
2026-11-19 11:52:48    1.3 KiB knowledge_base/products/hydra-wash-shampoo.md
2026-11-19 11:52:48    1.4 KiB knowledge_base/products/repair-bond-oil.md
2026-11-19 11:52:48    1.3 KiB knowledge_base/products/scalp-reset-serum.md
2026-11-19 11:52:48    1.4 KiB knowledge_base/products/silk-slip-conditioner.md
2026-11-19 11:52:48    1.3 KiB knowledge_base/products/weekly-repair-mask.md

Total Objects: 11
   Total Size: 18.4 KiB

Local .md files: 11


## Part 4: Amazon Bedrock Knowledge base

Create the knowledge base with out of the box values and test it with an eval dataset.

In [ ]:
EVAL_SET = [
    {"q": "Is the bond oil safe for bleached hair?", "relevant": ["repair-bond-oil"], "type": "lexical"},
    {"q": "How much does shipping cost?", "relevant": ["faq"], "type": "lexical"},
    {"q": "Up to what temperature does the heat protectant work?", "relevant": ["heat-shield-mist"], "type": "lexical"},
    {"q": "Is niacinamide in any of your products?", "relevant": ["scalp-reset-serum"], "type": "lexical"},
    {"q": "Does the conditioner contain silicones?", "relevant": ["silk-slip-conditioner"], "type": "lexical"},
    {"q": "Is there a holiday sale?", "relevant": ["aws-developers-discount"], "type": "lexical"},
    {"q": "Are your products tested on animals?", "relevant": ["faq"], "type": "lexical"},
    {"q": "How often should I use the deep conditioning mask?", "relevant": ["weekly-repair-mask"], "type": "lexical"},
    {"q": "What routine do you recommend for fine hair?", "relevant": ["hair-type-guide"], "type": "lexical"},
    {"q": "What should I spray on before using my flat iron?", "relevant": ["heat-shield-mist"], "type": "paraphrase"},
    {"q": "My scalp gets greasy really fast, what can help?", "relevant": ["scalp-reset-serum", "hair-type-guide"], "type": "paraphrase"},
    {"q": "Do you have any promo codes?", "relevant": ["aws-developers-discount"], "type": "paraphrase"},
    {"q": "Which product gives my curls hold without making them stiff?", "relevant": ["air-dry-styling-gel", "define-curl-cream"], "type": "paraphrase"},
    {"q": "Can I get my money back if I already opened the bottle?", "relevant": ["faq"], "type": "paraphrase"},
    {"q": "How many days do I have to send something back?", "relevant": ["faq"], "type": "paraphrase"},
    {"q": "Which shampoo won't wash out my dye job?", "relevant": ["hydra-wash-shampoo"], "type": "paraphrase"},
    {"q": "When will I see the money from my return on my card?", "relevant": ["faq"], "type": "paraphrase"},
    {"q": "Will the product with argan oil make thin hair look oily?", "relevant": ["repair-bond-oil"], "type": "paraphrase"},
]
NOISE = round(1 / len(EVAL_SET), 3)  # one question's worth of a metric


def approx_tokens(text: str) -> int:
    return round(len(text) / 4)

def run(search_fn, k: int = 10) -> list[dict]:
    """Run every eval question once and keep the full ranking, so metrics can be read at any k."""
    rows = []
    for item in EVAL_SET:
        hits = search_fn(item["q"], k)
        rows.append({"q": item["q"], "type": item["type"], "relevant": set(item["relevant"]),
                     "ranking": [h["doc_id"] for h in hits], "tokens": [approx_tokens(h["text"]) for h in hits]})
    return rows


def metrics_at(rows: list[dict], k: int) -> pd.Series:
    """hit@1, hit@k, recall@k (share of relevant docs in the top k), MRR (within the full ranking), tokens in the top k."""
    out = []
    for r in rows:
        first = next((i + 1 for i, d in enumerate(r["ranking"]) if d in r["relevant"]), None)
        out.append({
            "hit@1": float(first == 1),
            "hit@k": float(first is not None and first <= k),
            "recall@k": len(r["relevant"] & set(r["ranking"][:k])) / len(r["relevant"]),
            "mrr": 1 / first if first else 0.0,
            "ctx_tokens@k": sum(r["tokens"][:k]),
        })
    return pd.DataFrame(out).mean().round(3)


print(f"{len(EVAL_SET)} questions; one question = {NOISE} of a metric")

18 questions; one question = 0.056 of a metric


In [26]:
KB_ID = "GAPCVFJ2YM"
kb_client = boto3.client("bedrock-agent-runtime", region_name=REGION)

In [27]:
def retrieve(query: str, k: int = 10) -> list[dict]:
    """Search the knowledge base and return hits in the shape run() expects."""
    response = kb_client.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "managedSearchConfiguration": {
                "numberOfResults": k,
            }
        },
    )
    return [
        {
            "doc_id": Path(item["documentId"]).stem,  # e.g. "repair-bond-oil", matches EVAL_SET
            "text": item["content"]["text"],
            "score": item.get("score"),
        }
        for item in response["retrievalResults"]
    ]

rows = run(retrieve, k=10)
metrics_at(rows, k=5)


hit@1              1.000
hit@k              1.000
recall@k           1.000
mrr                1.000
ctx_tokens@k    1916.722
dtype: float64

## Cleanup

Deleting the bucket removes every object in it. Uncomment only when you are done with this bucket.

In [28]:
run_aws("s3", "rb", f"s3://{BUCKET}", "--force", "--region", REGION)
print("Deleted bucket", BUCKET)

Deleted bucket product-information-kb-data-820eb591
